In [0]:
from pyspark.sql.functions import *
from pyspark.sql import Window
from delta.tables import DeltaTable

bronze_table = "data_modelling_demo_catalog.bronze.scd_1_output"
silver_table = "data_modelling_demo_catalog.silver.scd_1_output"

dbutils.widgets.text("insert_timestamp", "", "Insert Timestamp (YYYY-MM-DD HH:MM:SS)")
insert_timestamp = dbutils.widgets.get("insert_timestamp")

# 1. read bronze table
if not insert_timestamp:
    df = spark.read.table(bronze_table)
    load_mode = "full"
else:
    df = (spark.read.table(bronze_table)
          .filter(col("ingestion_time") > to_timestamp(lit(insert_timestamp))))
    load_mode = "append"

# 2. do the transformations
transformed_df = (
    df.withColumn("first_name", initcap(trim(col("first_name"))))
      .withColumn("email", lower(trim(col("email"))))
      .withColumn("registration_date", to_date(col("registration_date"), "dd-MMM-yyyy"))
)

# 3. do the data quality checks
valid_conditions = (
    col("customer_id").isNotNull()
    & col("first_name").isNotNull()
    & (trim(col("first_name")) != "")
    & col("email").rlike(r"^[^@\s]+@[^@\s]+\.[^@\s]+$")
    & col("registration_date").isNotNull()
)

valid_df = transformed_df.filter(valid_conditions)
invalid_df = transformed_df.filter(~valid_conditions)
print(f"Count of invalid entries: {invalid_df.count()}")

# 4. keep only the latest record per customer (in case we have duplicate entries coming in from bronze keep the latest entry only)
window_spec = Window.partitionBy("customer_id").orderBy(col("ingestion_time").desc())
valid_df = (
    valid_df
    .withColumn("_row_num", row_number().over(window_spec))
    .filter(col("_row_num") == 1)
    .drop("_row_num")
)

valid_df=valid_df.withColumn("processed_time", current_timestamp())
# 5. write to silver
if load_mode == "full" or not spark.catalog.tableExists(silver_table):
    (valid_df.write.format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(silver_table))
    print(f"Full load done from bronze to silver")
else:
    silver_dt = DeltaTable.forName(spark, silver_table)
    (silver_dt.alias("t")
        .merge(valid_df.alias("s"), "t.customer_id = s.customer_id")
        .whenMatchedUpdateAll(condition="s.ingestion_time > t.ingestion_time")
        .whenNotMatchedInsertAll()
        .execute())

    # read merge metrics from the latest commit
    metrics = silver_dt.history(1).select("operationMetrics").collect()[0][0]
    print(f"Merge complete. Inserted: {metrics.get('numTargetRowsInserted')}, "
          f"Updated: {metrics.get('numTargetRowsUpdated')}")


In [0]:
%sql
select * from data_modelling_demo_catalog.silver.scd_1_output;
